<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_5_Specialization/Track_2_MLOps/docker_model_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deploying ML Models with Docker
## Learning Objectives
- Package an ML model as a Docker container
- Build a REST API with FastAPI
- Containerize and run the service


In [ ]:
# ML model training and serialization
import pickle
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f'Test accuracy: {model.score(X_test, y_test):.4f}')

# Save model
with open('iris_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved to iris_model.pkl')

In [ ]:
# FastAPI app (save as app.py)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title='Iris Classifier API')

with open('iris_model.pkl', 'rb') as f:
    model = pickle.load(f)

class IrisFeatures(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post('/predict')
def predict(features: IrisFeatures):
    X = [[features.sepal_length, features.sepal_width,
          features.petal_length, features.petal_width]]
    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0].max()
    species = ['setosa', 'versicolor', 'virginica'][pred]
    return {'species': species, 'confidence': round(float(proba), 4)}
'''
print('FastAPI app code:')
print(fastapi_code)

In [ ]:
# Dockerfile content
dockerfile = '''
FROM python:3.9-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''
requirements = 'fastapi\nuvicorn[standard]\nscikit-learn\nnumpy\npandas\n'

print('Dockerfile:')
print(dockerfile)
print('requirements.txt:')
print(requirements)

In [ ]:
# Docker commands
docker_commands = [
    'docker build -t iris-classifier:v1 .',
    'docker run -d -p 8000:8000 --name iris-api iris-classifier:v1',
    'docker ps  # verify container is running',
    'curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" -d \'{{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}}\'',
    'docker stop iris-api && docker rm iris-api'
]
print('Docker workflow:')
for cmd in docker_commands:
    print(f'  $ {cmd}')

## Exercises
1. Build the Docker image and test the API with `curl` or Postman.
2. Add a `/health` endpoint and `/metrics` endpoint.
3. Push the image to Docker Hub and deploy on a free cloud service.


## Summary
- Docker packages your model with all dependencies for reproducible deployment.
- FastAPI provides fast, auto-documented REST APIs.
- Container registries (Docker Hub, ECR) enable sharing and deployment.
